In [1]:
%pip -q install "numpy>=1.26,<3" "pandas>=2,<3" "scikit-learn>=1.5,<1.8" "xgboost==3.0.5" "shap==0.48.0" "optuna==4.5.0" openpyxl matplotlib scipy joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.5/351.5 MB 4.7 MB/s eta 0:00:00


In [ ]:
from google.colab import files
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

print('Upload the panel XLSX and the three city recovery-result CSVs together.')
uploaded = files.upload()
panel_candidates = [name for name in uploaded if name.lower().endswith('.xlsx')]
if len(panel_candidates) != 1:
    raise ValueError('Upload exactly one community-panel XLSX.')
PANEL_FILE = Path(panel_candidates[0])
BURDEN_FILES = [Path(name) for name in uploaded if name.lower().endswith('.csv')]
if len(BURDEN_FILES) != 3:
    raise ValueError('Upload three result CSVs: Shenyang, Harbin, and Changchun.')
REQUIRE_COMPLETE_BURDEN = False
OUTPUT = Path('pooled_trr_results')
OUTPUT.mkdir(exist_ok=True)


Upload the panel XLSX and the three city recovery-result CSVs together.


In [ ]:
EXPECTED_COUNTS = {'Shenyang': 2700, 'Harbin': 1977, 'Changchun': 1689}

def validate_city_result_files(panel_file, burden_files):
    sheets = pd.read_excel(panel_file, sheet_name=None)
    city_ids = {}
    for sheet, table in sheets.items():
        if 'Community_ID' not in table:
            continue
        city_series = table['City'] if 'City' in table else pd.Series(sheet, index=table.index)
        for city in city_series.unique():
            city_ids.setdefault(city, set()).update(table.loc[city_series == city, 'Community_ID'])
    matched = {}
    for path in burden_files:
        result = pd.read_csv(path)
        if not {'Community_ID', 'Recovery_burden_B'} <= set(result):
            raise ValueError(f'{path}: requires Community_ID and Recovery_burden_B.')
        if result.Community_ID.isna().any() or result.Community_ID.duplicated().any():
            raise ValueError(f'{path}: missing or duplicate community IDs.')
        ids = set(result.Community_ID)
        cities = [city for city, expected_ids in city_ids.items() if ids == expected_ids]
        if len(cities) != 1:
            raise ValueError(f'{path}: IDs must match exactly one complete city panel.')
        city = cities[0]
        if city in matched:
            raise ValueError(f'More than one result CSV supplied for {city}.')
        matched[city] = str(path)
    if set(matched) != set(EXPECTED_COUNTS):
        raise ValueError('Result files must cover Shenyang, Harbin, and Changchun.')
    print('City result files:', json.dumps(matched, indent=2))
    return matched

def prepare_pooled_trr(panel_file, burden_files, output, require_complete=True):
    sheets = pd.read_excel(panel_file, sheet_name=None)
    frames = []
    for sheet_name, frame in sheets.items():
        if 'Community_ID' not in frame:
            continue
        frame = frame.rename(columns={'District_romanized': 'District'}).copy()
        if 'City' not in frame:
            frame['City'] = sheet_name
        frames.append(frame)
    if not frames:
        raise ValueError('No panel sheets with Community_ID were found.')
    panel = pd.concat(frames, ignore_index=True)
    if panel.Community_ID.isna().any() or panel.Community_ID.duplicated().any():
        raise ValueError('Panel Community_ID must be globally unique and nonmissing.')
    counts = panel.groupby('City').size().to_dict()
    if counts != EXPECTED_COUNTS:
        raise ValueError(f'Unexpected city counts: {counts}; expected {EXPECTED_COUNTS}.')
    panel['TRR_supplied'] = panel['TRR'] if 'TRR' in panel else np.nan

    if burden_files:
        parts = []
        for path in burden_files:
            table = pd.read_csv(path)
            required = {'Community_ID', 'Recovery_burden_B'}
            if not required <= set(table):
                raise ValueError(f'{path}: missing Community_ID or Recovery_burden_B.')
            parts.append(table[['Community_ID', 'Recovery_burden_B']])
        burdens = pd.concat(parts, ignore_index=True)
        if burdens.Community_ID.isna().any() or burdens.Community_ID.duplicated().any():
            raise ValueError('Burden files contain missing or duplicate Community_ID.')
        if not set(burdens.Community_ID) <= set(panel.Community_ID):
            raise ValueError('Some burden IDs do not belong to the supplied panel.')
        panel = panel.drop(columns=['Recovery_burden_B'], errors='ignore').merge(
            burdens, on='Community_ID', how='left', validate='one_to_one')
        burden_source = 'Uploaded original calculation outputs'
    elif 'Recovery_burden_B' in panel:
        burden_source = 'Original recovery burden in panel'
    elif {'T90_hours', 'E_c'} <= set(panel):
        t = pd.to_numeric(panel['T90_hours'], errors='raise')
        e = pd.to_numeric(panel['E_c'], errors='raise')
        if (t.dropna() < 0).any() or (e.dropna() <= 0).any():
            raise ValueError('T90_hours must be nonnegative and E_c must be positive.')
        panel['Recovery_burden_B'] = t / e
        burden_source = 'Panel T90_hours divided by panel E_c'
    else:
        raise ValueError(
            'The supplied panel has TRR scores but no original Recovery_burden_B. '
            'Upload the three-city calculation outputs or add the original burden column. '
            'Existing city-level TRR scores cannot supply the original pooled bounds.')

    b = pd.to_numeric(panel['Recovery_burden_B'], errors='raise')
    if np.isinf(b).any() or (b.dropna() < 0).any():
        raise ValueError('Recovery burdens must be finite and nonnegative when available.')
    panel['Recovery_burden_B'] = b
    missing = panel.loc[b.isna(), ['Community_ID', 'City']]
    missing.to_csv(output / 'Missing_burden_communities.csv', index=False)
    if require_complete and len(missing):
        raise ValueError(
            f'{len(missing)} communities have missing original burdens. '
            'The manuscript-wide 6366-community normalization cannot be completed. '
            'See Missing_burden_communities.csv.')
    if panel.loc[b.notna(), 'City'].nunique() != 3:
        raise ValueError('Valid burdens must cover all three cities.')

    # Pool the original burdens BEFORE splitting or fitting any model.
    B_min = float(b.min())
    B_max = float(b.max())
    if not np.isfinite(B_min) or not np.isfinite(B_max) or B_max <= B_min:
        raise ValueError('Pooled burden bounds must be finite and distinct.')

    # One reverse min-max normalization for the entire analytical sample.
    panel['TRR'] = 100.0 * (1.0 - (b - B_min) / (B_max - B_min))
    panel['Target_TRR'] = panel['TRR']
    panel['Target_source'] = 'Pooled_reverse_normalization_from_original_burden'
    valid = panel.loc[b.notna()].sort_values('Recovery_burden_B')
    assert valid.TRR.between(-1e-8, 100 + 1e-8).all()
    assert np.all(np.diff(valid.TRR.to_numpy()) <= 1e-8)
    panel['Difference_from_supplied_TRR'] = panel.TRR - pd.to_numeric(
        panel.TRR_supplied, errors='raise')
    panel.to_csv(output / 'Community_panel_pooled_TRR.csv', index=False, float_format='%.10f')
    panel[['Community_ID', 'TRR']].to_csv(
        output / 'Pooled_TRR.csv', index=False, float_format='%.10f')
    panel[['Community_ID', 'Recovery_burden_B']].to_csv(
        output / 'Original_recovery_burdens.csv', index=False, float_format='%.17g')
    summary = panel.groupby('City').agg(
        Communities=('Community_ID', 'size'), Valid_burdens=('Recovery_burden_B', 'count'),
        TRR_min=('TRR', 'min'), TRR_max=('TRR', 'max'), TRR_mean=('TRR', 'mean'))
    summary.to_csv(output / 'City_TRR_summary.csv', float_format='%.10f')
    audit = {
        'formula': 'TRR = 100 * [1 - (B - B_min) / (B_max - B_min)]',
        'burden_definition': 'B = T90_hours / E_c',
        'burden_source': burden_source,
        'B_min': B_min, 'B_max': B_max,
        'communities_in_panel': len(panel), 'communities_with_burden': int(b.notna().sum()),
        'missing_burdens': len(missing), 'city_counts': counts,
        'normalization_scope': 'All 6366 communities' if b.notna().all() else 'Reduced three-city sample',
        'same_bounds_for_all_cities': True,
        'normalization_completed_before_split': True,
        'input_sha256': {str(p): hashlib.sha256(Path(p).read_bytes()).hexdigest()
                         for p in [panel_file, *burden_files]},
    }
    (output / 'Global_normalization_audit.json').write_text(json.dumps(audit, indent=2))
    return panel, audit, summary

city_result_files = validate_city_result_files(PANEL_FILE, BURDEN_FILES)
panel, normalization_audit, city_summary = prepare_pooled_trr(
    PANEL_FILE, BURDEN_FILES, OUTPUT, REQUIRE_COMPLETE_BURDEN)
display(city_summary)
print(json.dumps(normalization_audit, indent=2))
display(panel[['Community_ID', 'City', 'Recovery_burden_B', 'TRR']].head(10))
normalization_audit['city_result_files'] = city_result_files
(OUTPUT / 'Global_normalization_audit.json').write_text(json.dumps(normalization_audit, indent=2))
print('Usable outcome rows:', int(panel.TRR.notna().sum()))


In [ ]:
"""Train/test separation for community TRR prediction. No outcomes are fabricated.

CV preprocessing and tuning use the training partition only. Held-out data are
used once for final metrics, SHAP interpretation and partial dependence.
"""
from pathlib import Path
import argparse,json,hashlib,platform,warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split,KFold,GroupShuffleSplit,GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

ALL_FEATURES=['WD','Dis_W','PAB','RD','BH','PAR','DL','DM','PAG','SWC','H_D','DB','Dis_M','Dis_B','NDVI','PCI','GTV','GGV','NL','PD','Slope','H_Price']
LOG_FEATURES=['BH','Dis_M','Dis_B','DB','H_D','Slope']
MANUSCRIPT_EXCLUDED=['NDVI','PCI','NL','Slope','Dis_B','PAB','H_D']
FEATURES=[f for f in ALL_FEATURES if f not in MANUSCRIPT_EXCLUDED]
DIMENSION={f:dimension for dimension,cols in {'Blue':['WD','Dis_W','PAB'],'Grey':['RD','BH','PAR','DL','DM','PAG','SWC','H_D','DB','Dis_M','Dis_B'],'Green':['NDVI','PCI','GTV','GGV'],'Control':['NL','PD','Slope','H_Price']}.items() for f in cols}

TYPES=['CHC','UC','OC','UVC']

def reverse_normalize_burden(burden,B_min,B_max):
    """TRR = 100 * [1 - (B - B_min) / (B_max - B_min)]."""
    if not(np.isfinite(B_min) and np.isfinite(B_max) and B_max>B_min):raise ValueError('Invalid normalization bounds')
    b=np.asarray(burden,dtype=float);valid=np.isfinite(b)
    if np.any(b[valid]<B_min-1e-8) or np.any(b[valid]>B_max+1e-8):raise ValueError('Normalization bounds do not cover the supplied burdens')
    return 100.0*(1.0-(b-B_min)/(B_max-B_min))

def verify_normalization(panel,args,out):
    # Reading a 0-100 column does not itself verify its historical transformation.
    report={'formula':'TRR = 100 * [1 - (B - B_min) / (B_max - B_min)]','model_target':'TRR read unchanged from the supplied panel','model_target_scale':'0-100 TRR points','repeated_normalization':False,'status':'Not_verifiable_from_panel_alone','reason':'The panel contains no original recovery burden or normalization bounds.'}
    if args.burden_file:
        burden=pd.read_csv(args.burden_file)
        required={'Community_ID','Recovery_burden_B'}
        if not required<=set(burden) or burden.Community_ID.duplicated().any():raise ValueError('Burden CSV requires unique Community_ID and Recovery_burden_B')
        if args.global_min is None or args.global_max is None:raise ValueError('Normalization verification requires the actual global B_min and B_max')
        merged=panel[['Community_ID','Target_TRR']].merge(burden[list(required)],on='Community_ID',how='left',validate='one_to_one')
        merged['TRR_from_burden']=reverse_normalize_burden(pd.to_numeric(merged.Recovery_burden_B,errors='raise'),args.global_min,args.global_max)
        valid=merged[['Target_TRR','TRR_from_burden']].notna().all(axis=1)
        if not valid.any():raise ValueError('No paired TRR and burden values to verify')
        merged['Absolute_difference']=abs(merged.Target_TRR-merged.TRR_from_burden);merged.to_csv(out/'Reverse_normalization_comparison.csv',index=False)
        error=float(merged.loc[valid,'Absolute_difference'].max());passed=error<=1e-6
        report.update(status='Verified_for_paired_rows' if passed else 'Mismatch',B_min=args.global_min,B_max=args.global_max,verified_rows=int(valid.sum()),unverified_rows=int((panel.Target_TRR.notna()&~valid).sum()),maximum_absolute_difference=error)
        (out/'Normalization_verification.json').write_text(json.dumps(report,indent=2))
        if not passed:raise ValueError('Panel TRR does not match supplied reverse-normalization calculation')
    (out/'Normalization_verification.json').write_text(json.dumps(report,indent=2))

def read_panel(path,trr_source='original',trr_file=None,type_file=None):
    sheets=pd.read_excel(path,sheet_name=None);frames=[]
    for name,d in sheets.items():
        if 'Community_ID' not in d:continue
        d=d.rename(columns={'District_romanized':'District'}).copy()
        if 'City' not in d:d['City']=name
        frames.append(d)
    if not frames:raise ValueError('No community panel sheet')
    panel=pd.concat(frames,ignore_index=True)
    if panel.Community_ID.isna().any() or panel.Community_ID.duplicated().any():raise ValueError('Community IDs must be unique and nonmissing')
    if type_file:
        labels=pd.read_csv(type_file)[['Community_ID','Community_type']]
        if labels.Community_ID.isna().any() or labels.Community_ID.duplicated().any():raise ValueError('Original type labels require unique community IDs')
        if 'Community_type' in panel:panel=panel.drop(columns='Community_type')
        panel=panel.merge(labels,on='Community_ID',how='left',validate='one_to_one')
        if panel.Community_type.isna().any():raise ValueError('Some updated communities have no original type label')
    missing=[x for x in FEATURES if x not in panel]
    if missing:raise ValueError('Missing predictors: '+str(missing))
    for col in FEATURES:panel[col]=pd.to_numeric(panel[col],errors='raise')
    panel[FEATURES]=panel[FEATURES].replace([np.inf,-np.inf],np.nan)
    if trr_source=='original':
        panel['Target_TRR']=pd.to_numeric(panel.TRR,errors='raise');panel['Target_source']='Panel_TRR'
    else:
        if trr_file is None:raise ValueError('Recalculated mode requires a CSV containing pooled TRR')
        outcomes=pd.read_csv(trr_file)
        if not {'Community_ID','TRR'}<=set(outcomes):raise ValueError('Recalculated CSV must contain Community_ID and TRR; Harbin-only normalization is not accepted')
        if outcomes.Community_ID.duplicated().any():raise ValueError('Duplicated outcome IDs')
        unused=set(outcomes.Community_ID)-set(panel.Community_ID)
        if unused:raise ValueError('Unknown outcome IDs: '+str(list(unused)[:5]))
        panel=panel.merge(outcomes[['Community_ID','TRR']].rename(columns={'TRR':'Target_TRR'}),on='Community_ID',how='left',validate='one_to_one')
        panel['Target_TRR']=pd.to_numeric(panel.Target_TRR,errors='raise');panel['Target_source']='Recalculated_pooled_TRR'
        # Original and recalculated outcomes are never mixed to fill gaps.
    good=panel.Target_TRR.dropna()
    if not good.between(0,100).all():raise ValueError('TRR must be on the 0-100 scale')
    if panel.Target_TRR.notna().sum()==0:raise ValueError('No valid outcomes; supply pooled TRR or choose original mode')
    panel['Outcome_available']=panel.Target_TRR.notna()
    return panel

def log_transform_predictors(panel):
    panel=panel.copy()
    for feature in ALL_FEATURES:
        panel[feature]=pd.to_numeric(panel[feature],errors='raise')
        panel[feature]=panel[feature].replace([np.inf,-np.inf],np.nan)
    for feature in LOG_FEATURES:
        values=panel[feature]
        if (values.dropna()<=0).any():
            raise ValueError(f'{feature}: natural logarithm requires positive values; no offset is assumed.')
        panel[feature+'_raw']=values
        panel[feature]=np.log(values)
    return panel

def pearson_report(panel,folder,label):
    from scipy.stats import pearsonr
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    columns=ALL_FEATURES+['Target_TRR']
    r=pd.DataFrame(np.nan,index=columns,columns=columns)
    pvalues=r.copy();counts=pd.DataFrame(0,index=columns,columns=columns)
    for i,a in enumerate(columns):
        for b in columns[i:]:
            pair=panel[[a,b]] if a!=b else panel[[a]].copy()
            valid=panel[a].notna() & panel[b].notna()
            av=panel.loc[valid,a];bv=panel.loc[valid,b]
            counts.loc[a,b]=counts.loc[b,a]=int(valid.sum())
            if valid.sum()>=3 and av.nunique()>1 and bv.nunique()>1:
                corr,prob=pearsonr(av,bv)
                r.loc[a,b]=r.loc[b,a]=float(corr)
                pvalues.loc[a,b]=pvalues.loc[b,a]=float(prob)
    r.to_csv(folder/'Pearson_r.csv',float_format='%.10f')
    pvalues.to_csv(folder/'Pearson_p.csv',float_format='%.10g')
    counts.to_csv(folder/'Pearson_pairwise_n.csv')
    outcome=pd.DataFrame({'Feature':ALL_FEATURES,'Dimension':[DIMENSION[f] for f in ALL_FEATURES],
        'r_with_TRR':[r.loc[f,'Target_TRR'] for f in ALL_FEATURES],
        'p_with_TRR':[pvalues.loc[f,'Target_TRR'] for f in ALL_FEATURES],
        'Paired_n':[counts.loc[f,'Target_TRR'] for f in ALL_FEATURES],
        'Retained_in_manuscript_model':[f in FEATURES for f in ALL_FEATURES]})
    outcome['Significant_p_lt_005']=outcome.p_with_TRR<.05
    outcome.to_csv(folder/'Pearson_with_TRR.csv',index=False)
    high=[]
    for i,a in enumerate(ALL_FEATURES):
        for b in ALL_FEATURES[i+1:]:
            if DIMENSION[a]!=DIMENSION[b] or not np.isfinite(r.loc[a,b]) or abs(r.loc[a,b])<=.7:continue
            ar=abs(r.loc[a,'Target_TRR']);br=abs(r.loc[b,'Target_TRR'])
            preferred=a if ar>=br else b
            high.append({'Feature_1':a,'Feature_2':b,'r':r.loc[a,b],'p':pvalues.loc[a,b],
                'Paired_n':counts.loc[a,b],'Stronger_abs_r_with_TRR':preferred,
                'Both_in_manuscript_model':a in FEATURES and b in FEATURES})
    pd.DataFrame(high,columns=['Feature_1','Feature_2','r','p','Paired_n','Stronger_abs_r_with_TRR','Both_in_manuscript_model']).to_csv(folder/'High_correlation_same_dimension.csv',index=False)
    fig,ax=plt.subplots(figsize=(14,12));im=ax.imshow(r.to_numpy(),vmin=-1,vmax=1,cmap='RdBu_r')
    labels=[f'ln({f})' if f in LOG_FEATURES else 'TRR' if f=='Target_TRR' else f for f in columns]
    ax.set_xticks(range(len(columns)),labels=labels,rotation=90);ax.set_yticks(range(len(columns)),labels=labels)
    for i,a in enumerate(columns):
        for j,b in enumerate(columns):
            if not np.isfinite(r.loc[a,b]):continue
            prob=pvalues.loc[a,b]
            # Preserve the manuscript Fig. 8 legend, although its star convention is unusual.
            stars='*' if prob<.01 else '**' if prob<.05 else ''
            ax.text(j,i,f'{r.loc[a,b]:.2f}{stars}',ha='center',va='center',fontsize=6)
    ax.set_title(label+' | * p < 0.01; ** p < 0.05')
    fig.colorbar(im,ax=ax,label='Pearson r');fig.tight_layout();fig.savefig(folder/'Pearson_heatmap.png',dpi=300);plt.close(fig)
    return outcome

def run_descriptive_pearson(panel,out):
    usable=panel[panel.Target_TRR.notna()].copy()
    counts=panel.groupby('Community_type').agg(Panel_n=('Community_ID','size'),Valid_TRR_n=('Target_TRR','count'))
    counts.to_csv(out/'Community_type_counts.csv')
    for label in ['Pooled']+TYPES:
        subset=usable if label=='Pooled' else usable[usable.Community_type==label]
        pearson_report(subset,out/'Pearson_descriptive'/label,label+' descriptive analysis')
    (out/'Preprocessing_specification.json').write_text(json.dumps({
        'candidate_predictors':ALL_FEATURES,'natural_log_predictors':LOG_FEATURES,
        'excluded_predictors_from_manuscript_text':MANUSCRIPT_EXCLUDED,'model_predictors':FEATURES,
        'selection_mode':'Fixed manuscript list; current p values are reported, not forced to match manuscript',
        'screening_significance_assumption':0.05,
        'same_dimension_redundancy_rule_assumption':'absolute Pearson r > 0.7',
        'note':'Descriptive full-sample correlations never select model features. Training-only correlations are also exported.',
        'missing_data_rule':'Pairwise complete observations for correlations; training-fold median imputation for models',
        'model_input_scale':'Natural log for six named variables; remaining variables unchanged',
        'plot_scale':'Original feature units for SHAP dependence and PDP; SHAP values remain TRR points'
    },indent=2))

def spatial_groups(d,block_km=2):
    if d[['Longitude','Latitude']].isna().any().any():raise ValueError('Spatial split requires coordinates')
    x=d.Longitude.to_numpy()*111.32*np.cos(np.radians(d.Latitude.to_numpy()))
    y=d.Latitude.to_numpy()*110.57
    # Use a fixed city latitude for each city's x coordinate.
    for city in d.City.unique():
        mask=d.City.to_numpy()==city;x[mask]=d.Longitude.to_numpy()[mask]*111.32*np.cos(np.radians(d.loc[mask,'Latitude'].mean()))
    return np.array([f'{c}:{int(a)}:{int(b)}' for c,a,b in zip(d.City,np.floor(x/block_km),np.floor(y/block_km))])

def split_rows(d,mode='random',seed=20261014,block_km=2):
    idx=np.arange(len(d))
    if mode=='spatial':
        groups=spatial_groups(d,block_km)
        if len(set(groups))<7:raise ValueError('Too few spatial blocks for held-out test and five-fold CV')
        train,test=next(GroupShuffleSplit(n_splits=1,test_size=.3,random_state=seed).split(idx,groups=groups))
        return train,test,groups
    train,test=train_test_split(idx,test_size=.3,random_state=seed)
    return train,test,None

def metrics(y,p):
    return {'R2':float(r2_score(y,p)),'MAE':float(mean_absolute_error(y,p)),'RMSE':float(np.sqrt(mean_squared_error(y,p))),'MSE':float(mean_squared_error(y,p))}

def compare_baselines(X,y,train,test,folds,args,folder,xgb_score,community_ids):
    import optuna,joblib
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.tree import DecisionTreeRegressor
    from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
    from sklearn.svm import SVR
    scores=[dict(xgb_score,Estimator='XGBoost')]
    def make(kind,p):
        if kind=='DT':est=DecisionTreeRegressor(random_state=args.seed,**p)
        elif kind=='RF':est=RandomForestRegressor(random_state=args.seed,n_jobs=2,**p)
        elif kind=='GBDT':est=GradientBoostingRegressor(random_state=args.seed,**p)
        else:est=SVR(**p)
        steps=[('impute',SimpleImputer(strategy='median'))]
        if kind=='SVM':steps.append(('scale',StandardScaler()))
        return Pipeline(steps+[('model',est)])
    for kind in ['DT','RF','SVM','GBDT']:
        def objective(trial):
            if kind in ['DT','RF']:
                p={'max_depth':trial.suggest_int('max_depth',3,30),'min_samples_leaf':trial.suggest_int('min_samples_leaf',1,20),'min_samples_split':trial.suggest_int('min_samples_split',2,20)}
                if kind=='RF':p.update(n_estimators=trial.suggest_int('n_estimators',200,800,step=100),max_features=trial.suggest_float('max_features',.5,1))
            elif kind=='GBDT':p={'n_estimators':trial.suggest_int('n_estimators',100,800,step=100),'learning_rate':trial.suggest_float('learning_rate',.01,.3,log=True),'max_depth':trial.suggest_int('max_depth',2,8),'min_samples_leaf':trial.suggest_int('min_samples_leaf',1,20),'subsample':trial.suggest_float('subsample',.5,1)}
            else:p={'C':trial.suggest_float('C',.1,1000,log=True),'gamma':trial.suggest_float('gamma',1e-4,1,log=True),'epsilon':trial.suggest_float('epsilon',.01,5,log=True)}
            values=[]
            for a,b in folds:
                model=make(kind,p);model.fit(X.iloc[train[a]],y.iloc[train[a]]);values.append(np.sqrt(mean_squared_error(y.iloc[train[b]],model.predict(X.iloc[train[b]]))))
            return float(np.mean(values))
        study=optuna.create_study(direction='minimize',sampler=optuna.samplers.TPESampler(seed=args.seed));study.optimize(objective,n_trials=args.trials,n_jobs=1)
        model=make(kind,study.best_params);model.fit(X.iloc[train],y.iloc[train]);p=model.predict(X.iloc[test]);sc=metrics(y.iloc[test],p);sc.update(Estimator=kind,CV_RMSE=study.best_value,N_train=len(train),N_test=len(test));scores.append(sc)
        joblib.dump(model,folder/f'Baseline_{kind}.joblib');study.trials_dataframe().to_csv(folder/f'Tuning_{kind}.csv',index=False);(folder/f'Parameters_{kind}.json').write_text(json.dumps(study.best_params,indent=2))
        pd.DataFrame({'Community_ID':community_ids.iloc[test].to_numpy(),'Observed_TRR':y.iloc[test].to_numpy(),'Predicted_TRR':p}).to_csv(folder/f'Test_predictions_{kind}.csv',index=False)
    comparison=pd.DataFrame(scores);comparison['Best_training_CV']=comparison.CV_RMSE==comparison.CV_RMSE.min();comparison.to_csv(folder/'Five_model_comparison.csv',index=False)

def road_overlap(mapping,train_ids,test_ids):
    d=mapping[mapping.Community_ID.isin(set(train_ids)|set(test_ids))]
    a=set(d.loc[d.Community_ID.isin(train_ids),'Road_ID']);b=set(d.loc[d.Community_ID.isin(test_ids),'Road_ID'])
    return {'Training_roads':len(a),'Test_roads':len(b),'Shared_roads':len(a&b),'Fraction_test_roads_shared':len(a&b)/len(b) if b else None,'Communities_with_mapping':d.Community_ID.nunique(),'Communities_without_mapping':len(set(train_ids)|set(test_ids))-d.Community_ID.nunique()}

def train_one(d,name,args,road_map=None):
    import optuna,joblib,xgboost,shap,matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from xgboost import XGBRegressor
    folder=Path(args.out)/name;folder.mkdir(parents=True,exist_ok=True)
    if len(d)<50:raise ValueError(f'{name}: fewer than 50 usable rows')
    d=d.reset_index(drop=True);train,test,groups=split_rows(d,args.split,args.seed,args.block_km)
    pearson_report(d.iloc[train],folder/'Pearson_training',name+' training only')
    X=d[FEATURES];y=d.Target_TRR
    if X.iloc[train].isna().all(axis=0).any():raise ValueError('An entire training predictor is missing')
    if groups is None:folds=list(KFold(n_splits=5,shuffle=True,random_state=args.seed).split(train))
    else:
        if len(set(groups[train]))<5:raise ValueError('Training partition has fewer than five blocks')
        folds=list(GroupKFold(n_splits=5).split(train,groups=groups[train]))
    identity_columns=[c for c in ['Community_ID','City','Community_type'] if c in d]
    membership=d[identity_columns].copy();membership['Partition']='test';membership.loc[train,'Partition']='train'
    if groups is not None:membership['Spatial_block']=groups
    membership.to_csv(folder/'Split_membership.csv',index=False)
    assert not(set(d.Community_ID.iloc[train])&set(d.Community_ID.iloc[test]))
    fixed=dict(objective='reg:squarederror',tree_method='hist',eval_metric='rmse',random_state=args.seed,n_jobs=2)
    def objective(trial):
        p={'learning_rate':trial.suggest_float('learning_rate',.01,.30,log=True),'max_depth':trial.suggest_int('max_depth',3,15),'min_child_weight':trial.suggest_int('min_child_weight',1,20),'n_estimators':trial.suggest_int('n_estimators',300,1500,step=100),'subsample':trial.suggest_float('subsample',.5,1),'colsample_bytree':trial.suggest_float('colsample_bytree',.5,1),'reg_lambda':trial.suggest_float('reg_lambda',0,20)}
        scores=[]
        for a,b in folds:
            fit_idx=train[a];validation_idx=train[b]
            imputer=SimpleImputer(strategy='median');xa=imputer.fit_transform(X.iloc[fit_idx]);xb=imputer.transform(X.iloc[validation_idx])
            model=XGBRegressor(**fixed,**p);model.fit(xa,y.iloc[fit_idx]);scores.append(np.sqrt(mean_squared_error(y.iloc[validation_idx],model.predict(xb))))
        trial.set_user_attr('fold_RMSE',[float(v) for v in scores]);return float(np.mean(scores))
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study=optuna.create_study(direction='minimize',sampler=optuna.samplers.TPESampler(seed=args.seed));study.optimize(objective,n_trials=args.trials,n_jobs=1)
    imputer=SimpleImputer(strategy='median');xtrain=imputer.fit_transform(X.iloc[train]);xtest=imputer.transform(X.iloc[test])
    model=XGBRegressor(**fixed,**study.best_params);model.fit(xtrain,y.iloc[train]);pred=model.predict(xtest)
    score=metrics(y.iloc[test],pred);score.update(Model=name,N_train=len(train),N_test=len(test),N_total=len(d),CV_RMSE=study.best_value,Target_source=d.Target_source.iloc[0],Split=args.split)
    pd.DataFrame([score]).to_csv(folder/'Test_metrics.csv',index=False);study.trials_dataframe().to_csv(folder/'Tuning_trials.csv',index=False)
    (folder/'Best_parameters.json').write_text(json.dumps(study.best_params,indent=2));joblib.dump({'model':model,'imputer':imputer,'features':FEATURES},folder/'Model.joblib');model.save_model(folder/'Model.json')
    predictions=d.iloc[test][identity_columns+['Target_TRR']].copy();predictions['Predicted_TRR']=pred;predictions['Residual']=predictions.Target_TRR-pred;predictions.to_csv(folder/'Test_predictions.csv',index=False)
    pd.DataFrame([metrics(y.iloc[train],model.predict(xtrain))]).to_csv(folder/'Training_metrics.csv',index=False)
    if args.compare:compare_baselines(X,y,train,test,folds,args,folder,score,d.Community_ID)
    pd.DataFrame({'Feature':FEATURES,'Training_median':imputer.statistics_}).to_csv(folder/'Training_imputation.csv',index=False)
    if road_map is not None:
        overlap=road_overlap(road_map,d.Community_ID.iloc[train],d.Community_ID.iloc[test]);(folder/'Road_overlap_diagnostic.json').write_text(json.dumps(overlap,indent=2))
        if overlap['Shared_roads']:warnings.warn(f'{name}: {overlap["Shared_roads"]} shared roads across train/test; random split is not an independent spatial validation')
    rng=np.random.default_rng(args.seed);chosen=np.sort(rng.choice(len(test),min(args.shap_n,len(test)),replace=False));xplot=pd.DataFrame(xtest[chosen],columns=FEATURES)
    explanation=shap.TreeExplainer(model,feature_perturbation='tree_path_dependent')(xplot)
    if explanation.values.shape!=(len(xplot),len(FEATURES)):raise ValueError('Unexpected SHAP dimensions')
    importance=pd.DataFrame({'Feature':FEATURES,'Mean_abs_SHAP':np.abs(explanation.values).mean(axis=0)}).sort_values('Mean_abs_SHAP',ascending=False);importance.to_csv(folder/'SHAP_importance.csv',index=False)
    values=pd.DataFrame(explanation.values,columns=FEATURES);values.insert(0,'Community_ID',d.iloc[test[chosen]].Community_ID.to_numpy());values.to_csv(folder/'SHAP_values.csv',index=False)
    plt.rcParams.update({'font.family':'serif','font.serif':['Times New Roman','DejaVu Serif'],'axes.facecolor':'white','figure.facecolor':'white'})
    shap.plots.beeswarm(explanation,max_display=len(FEATURES),show=False);plt.savefig(folder/'SHAP_beeswarm.png',dpi=300,bbox_inches='tight');plt.close('all')
    shap.plots.bar(explanation,max_display=len(FEATURES),show=False);plt.savefig(folder/'SHAP_importance.png',dpi=300,bbox_inches='tight');plt.close('all')
    physical_plot=xplot.copy()
    for f in LOG_FEATURES:
        if f in physical_plot:physical_plot[f]=np.exp(physical_plot[f])
    display_explanation=shap.Explanation(values=explanation.values,base_values=explanation.base_values,data=physical_plot.to_numpy(),feature_names=FEATURES)
    top=importance.Feature.iloc[:6].tolist()
    for feature in top:
        shap.plots.scatter(display_explanation[:,feature],show=False);plt.savefig(folder/f'SHAP_dependence_{feature}.png',dpi=300,bbox_inches='tight');plt.close('all')
        j=FEATURES.index(feature);grid=np.unique(np.quantile(xplot[feature],np.linspace(.02,.98,40)));response=[]
        for value in grid:
            changed=xplot.to_numpy().copy();changed[:,j]=value;response.append(float(model.predict(changed).mean()))
        physical_grid=np.exp(grid) if feature in LOG_FEATURES else grid
        pd.DataFrame({'Feature_value':physical_grid,'Mean_predicted_TRR':response}).to_csv(folder/f'PDP_{feature}.csv',index=False)
        fig,ax=plt.subplots(figsize=(5,3.5));ax.plot(physical_grid,response,color='black');ax.set(xlabel=feature,ylabel='Mean predicted TRR');fig.tight_layout();fig.savefig(folder/f'PDP_{feature}.png',dpi=300);plt.close(fig)
    if args.interactions:
        ix=xplot.iloc[:min(200,len(xplot))];inter=shap.TreeExplainer(model,feature_perturbation='tree_path_dependent').shap_interaction_values(ix)
        magnitude=np.abs(np.asarray(inter)).mean(axis=0);np.fill_diagonal(magnitude,0)
        pd.DataFrame(magnitude,index=FEATURES,columns=FEATURES).to_csv(folder/'SHAP_interaction_strength.csv')
        np.savez_compressed(folder/'SHAP_interaction_values.npz',values=np.asarray(inter),features=np.array(FEATURES),community_ids=d.iloc[test[chosen[:len(ix)]]].Community_ID.to_numpy(dtype=str))
        pairs=sorted([(float(magnitude[a,b]),a,b) for a in range(len(FEATURES)) for b in range(a+1,len(FEATURES))],reverse=True)[:6]
        pd.DataFrame([{'Feature_1':FEATURES[a],'Feature_2':FEATURES[b],'Mean_abs_interaction':strength} for strength,a,b in pairs]).to_csv(folder/'Top_SHAP_interactions.csv',index=False)
        for strength,a,b in pairs:
            fig,ax=plt.subplots(figsize=(5,3.5));points=ax.scatter(physical_plot.iloc[:len(ix),a],np.asarray(inter)[:,a,b],c=physical_plot.iloc[:len(ix),b],cmap='viridis',s=14,alpha=.75)
            ax.set(xlabel=FEATURES[a],ylabel='SHAP interaction value');fig.colorbar(points,ax=ax,label=FEATURES[b]);fig.tight_layout();fig.savefig(folder/f'SHAP_interaction_{FEATURES[a]}_{FEATURES[b]}.png',dpi=300);plt.close(fig)
    score['Versions']={'xgboost':xgboost.__version__,'shap':shap.__version__,'optuna':optuna.__version__};(folder/'Run_summary.json').write_text(json.dumps(score,indent=2));return {k:v for k,v in score.items() if k!='Versions'}

def main(args):
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True);panel=read_panel(args.panel,args.trr_source,args.trr_file,args.type_file)
    panel=log_transform_predictors(panel)
    verify_normalization(panel,args,out)
    run_descriptive_pearson(panel,out)
    panel[['Community_ID','City','Target_TRR','Target_source','Outcome_available']].to_csv(out/'Model_target_values.csv',index=False,float_format='%.10f')
    panel[[c for c in ['Community_ID','City','Community_type','Target_source','Outcome_available'] if c in panel]].to_csv(out/'Sample_audit.csv',index=False)
    report={'rows':len(panel),'city_counts':panel.City.value_counts().to_dict(),'type_counts':panel.Community_type.value_counts().to_dict() if 'Community_type' in panel else None,'missing_outcomes':int(panel.Target_TRR.isna().sum()),'features':FEATURES,'panel_sha256':hashlib.sha256(Path(args.panel).read_bytes()).hexdigest(),'target_mode':args.trr_source,'split_mode':args.split,'test_fraction_requested':.3,'five_fold_tuning_training_only':True,'search_method':'Optuna TPE sequential Bayesian optimization','predictor_missingness':panel[FEATURES].isna().sum().to_dict()}
    (out/'Input_audit.json').write_text(json.dumps(report,indent=2));data=panel[panel.Outcome_available].copy()
    road_map=pd.read_csv(args.road_mapping) if args.road_mapping else None
    scores=[]
    if args.by_type and 'Community_type' not in data:raise ValueError('Community_type is absent; disable --by-type or provide validated type labels')
    if args.pooled:scores.append(train_one(data,'Pooled',args,road_map))
    if args.by_type:
        for typ in TYPES:scores.append(train_one(data[data.Community_type==typ],typ,args,road_map))
    if not scores:raise ValueError('Select --pooled and/or --by-type')
    pd.DataFrame(scores).to_csv(out/'All_test_metrics.csv',index=False)

def parser():
    p=argparse.ArgumentParser();p.add_argument('--panel',required=True);p.add_argument('--type-file');p.add_argument('--burden-file');p.add_argument('--global-min',type=float);p.add_argument('--global-max',type=float);p.add_argument('--out',default='model_output');p.add_argument('--trr-source',choices=['original','recalculated'],default='original');p.add_argument('--trr-file');p.add_argument('--split',choices=['random','spatial'],default='random');p.add_argument('--block-km',type=float,default=2);p.add_argument('--seed',type=int,default=20261014);p.add_argument('--trials',type=int,default=30);p.add_argument('--shap-n',type=int,default=1000);p.add_argument('--pooled',action='store_true');p.add_argument('--by-type',action='store_true');p.add_argument('--interactions',action='store_true');p.add_argument('--compare',action='store_true');p.add_argument('--road-mapping');return p



In [ ]:
from types import SimpleNamespace

config = SimpleNamespace(
    panel=str(PANEL_FILE), type_file=None,
    trr_source='recalculated', trr_file=str(OUTPUT / 'Pooled_TRR.csv'),
    burden_file=str(OUTPUT / 'Original_recovery_burdens.csv'),
    global_min=normalization_audit['B_min'], global_max=normalization_audit['B_max'],
    out=str(OUTPUT / 'models'), split='random', block_km=2.0,
    seed=20261014, trials=30, shap_n=1000,
    pooled=True, by_type=True, interactions=True, compare=True, road_mapping=None,
)
main(config)
display(pd.read_csv(OUTPUT / 'models/All_test_metrics.csv'))


In [ ]:
from IPython.display import Image, display

for model_name in ['Pooled', 'CHC', 'UC', 'OC', 'UVC']:
    folder = OUTPUT / 'models' / model_name
    for filename in ['SHAP_beeswarm.png', 'SHAP_importance.png']:
        path = folder / filename
        if path.exists():
            print(model_name, filename)
            display(Image(filename=str(path)))

print('Final 15 model predictors:', FEATURES)
display(pd.read_csv(OUTPUT / 'models/Community_type_counts.csv'))
for label in ['Pooled'] + TYPES:
    path = OUTPUT / 'models/Pearson_descriptive' / label / 'Pearson_heatmap.png'
    if path.exists():
        print(label, 'Pearson correlation')
        display(Image(filename=str(path)))


In [ ]:
import shutil

archive = shutil.make_archive('Three_city_TRR_results', 'zip', OUTPUT)
files.download(archive)
